# 2. Treinamento LoRA executado em CPU

O treino longo foi executado por `python -m scripts.experimento`, que chama o laço real em `scripts/treinar_lora.py`. Este notebook verifica e apresenta o histórico salvo. Abrir ou executar este notebook não repete o treino silenciosamente.

In [1]:
from pathlib import Path
import sys, json, hashlib
ROOT=Path.cwd()
if not (ROOT/'energia_clara').exists(): ROOT=ROOT.parent
assert (ROOT/'energia_clara').exists()
sys.path.insert(0,str(ROOT))
from energia_clara.config import read_json,read_jsonl
import pandas as pd
from IPython.display import display,Markdown
print('Python:',sys.version.split()[0])


Python: 3.14.7


In [2]:
history=read_json(ROOT/'avaliacao/historico_treino.json')
adapter=ROOT/'modelos/adaptador/adapter_model.safetensors'
assert hashlib.sha256(adapter.read_bytes()).hexdigest()==history['adaptador_sha256']
assert history['mascara_prompt_verificada']
assert not history['pilot']
print('Modelo:',history['modelo'])
print('Parâmetros treináveis:',history['parametros_treinaveis'])
print('Máscara do prompt e EOS verificados:',history['mascara_prompt_verificada'])
print('Melhor época pela validação:',history['melhor_epoca'])
print('Duração real em minutos:',round(history['duracao_s']/60,2))
print('RAM de pico amostrada do processo, GB:',round(history['ram_pico_gb'],2))
display(pd.DataFrame([history['config']]))
display(pd.DataFrame(history['epocas']))
display(pd.DataFrame(history['passos']).iloc[[0,1,2,-3,-2,-1]])

Modelo: HuggingFaceTB/SmolLM2-135M-Instruct
Parâmetros treináveis: 2442240
Máscara do prompt e EOS verificados: True
Melhor época pela validação: 1
Duração real em minutos: 23.97
RAM de pico amostrada do processo, GB: 2.13


,epocas,lote,acumulacao,learning_rate,rank,alpha,dropout,max_seq,seed,threads,dtype,device
0,4,2,2,0.0003,8,16,0.05,640,42,6,float32,cpu


,epoca,loss_treino,loss_validacao
0,1,0.712582,0.861185
1,2,0.143700,1.093267
2,3,0.025191,1.199443


,epoca,lote,passo,loss,segundos,ram_processo_gb
0,1,1,1,1.811509,4.445696,1.773106
1,1,2,2,1.746091,4.544159,1.436314
2,1,3,3,1.594163,4.450592,1.709095
297,3,98,298,0.021019,4.609545,0.879478
298,3,99,299,0.022152,4.297258,1.185818
299,3,100,300,0.011548,4.232274,0.831219


In [3]:
import plotly.graph_objects as go
epochs=history['epocas']
fig=go.Figure()
fig.add_scatter(x=[0]+[x['epoca'] for x in epochs],y=[history['loss_validacao_inicial']]+[x['loss_validacao'] for x in epochs],mode='lines+markers',name='Validação')
fig.add_scatter(x=[x['epoca'] for x in epochs],y=[x['loss_treino'] for x in epochs],mode='lines+markers',name='Treino')
fig.update_layout(title='Perda por época — seleção pelo menor valor de validação',xaxis_title='Época',yaxis_title='Cross-entropy')
display(fig)

A perda de treino caiu enquanto a validação piorou após a primeira época. O checkpoint da primeira época foi recarregado; não foi escolhido pelo teste. A revisão semântica da geração está no próximo notebook. A cópia exata do treinador usado inicialmente foi preservada em `avaliacao/protocolo_executado/`; a versão distribuída evita uma consulta remota opcional do PEFT no salvamento.